In [336]:
from email.quoprimime import header_check

import pandas as pd

In [337]:
data = pd.read_csv("ventes_brut .csv",sep=',' , index_col = 0, header = 0)

In [338]:
# Combien de lignes et de colonnes le dataset contient-il ?
data.shape

(228, 8)

In [339]:
# Quels sont les types de chaque colonne ? Sont-ils cohérents avec ce qu'ils devraient être ?

#Ils ne sont pas cohérents, prix devrait être en float data en date_Type
data.dtypes

date_vente           str
produit              str
categorie            str
prix_unitaire        str
quantite         float64
client               str
region               str
mode_paiement        str
dtype: object

In [340]:
#Que remarquez-vous en affichant les 10 premières lignes ?

#prix pas normé (signe euro parfois / , ou .)
#quantité valeurs abérantes (-1)
#catégorie pas normé (full Maj parfois / full lower case / une maj au début)
#même problème pour région
data.head(10)

,date_vente,produit,categorie,prix_unitaire,quantite,client,region,mode_paiement
id_vente,,,,,,,,
1,2025-10-07,Casque audio,électronique,NaN,-1.0,Client_1152,nouvelle aquitaine,Carte bancaire
2,2025-09-11,Casque audio,Electronique,"185,41 €",4.0,Client_1099,Île-de-France,Carte bancaire
3,28/11/2025,Stylo pack x10,fournitures,7.55,1.0,Client_1115,Auvergne-Rhône-Alpes,Espèces
4,2025-12-02,Chaise de bureau,mobilier,292.49,-1.0,Client_1128,IDF,Espèces
5,2025-12-10,Lampe LED,Mobilier,57.04,1.0,Client_1114,AURA,Chèque
6,2025-05-27,Bureau réglable,MOBILIER,"311,61 €",3.0,Client_1129,IDF,Chèque
7,2025-05-12,Stylo pack x10,Fourniture,7.72,3.0,Client_1165,Occitanie,Virement
8,2025-11-05,Ordinateur portable,électronique,"1187,64 €",4.0,Client_1004,Auvergne-Rhône-Alpes,Chèque
9,2025-05-11,Ordinateur portable,ELECTRONIQUE,1214.54,-1.0,Client_1101,NaN,Espèces


In [341]:
#Avant de corriger quoi que ce soit, quantifiez les problèmes :


In [342]:
#Nombre de valeurs manquantes par colonne
data.isnull().sum()

date_vente        0
produit           0
categorie        15
prix_unitaire    13
quantite         15
client            0
region           13
mode_paiement    13
dtype: int64

In [343]:
#Nombre de lignes strictement dupliquées
print(data.duplicated().sum())

8


In [344]:
#Valeurs uniques des colonnes categorie et region — que constatez-vous ?
#doublons car case différents (fournitures / Fourniture, mobilier / Mobilier, ELECTRONIQUE / Electronique, bretagne / Bretagne ...)
data['categorie'].unique()

<StringArray>
[ 'électronique', 'Electronique ',  'fournitures ',      'mobilier',
      'Mobilier',      'MOBILIER',    'Fourniture',  'ELECTRONIQUE',
   'Fournitures',  'Électronique',             nan]
Length: 11, dtype: str

In [345]:
data['region'].unique()


<StringArray>
[  'nouvelle aquitaine',        'Île-de-France', 'Auvergne-Rhône-Alpes',
                  'IDF',                 'AURA',            'Occitanie',
                    nan,             'bretagne',           'occitanie ',
        'ile de france',             'Bretagne',   'Nouvelle-Aquitaine']
Length: 12, dtype: str

In [346]:
#Valeurs uniques de quantite — voyez-vous des valeurs suspectes ?
#-1 est suscpect, dans le contexte de la données ça ne devrait pas exister
data['quantite'].unique()

array([-1.,  4.,  1.,  3.,  2.,  0., nan])

In [347]:
#Supprimez les lignes strictement dupliquées

data.drop_duplicates(inplace=True)

In [348]:
data.isna().sum()


date_vente        0
produit           0
categorie        13
prix_unitaire    13
quantite         13
client            0
region           13
mode_paiement    13
dtype: int64

In [349]:
# Traitez les valeurs manquantes de prix_unitaire et quantite

#on supprime les lignes avec quantité et prix en null car c'est une données nécéssaire qui permet de travailler
data = data.dropna(subset = ['quantite', 'prix_unitaire'])

In [350]:
data.isna().sum()


date_vente        0
produit           0
categorie        12
prix_unitaire     0
quantite          0
client            0
region           13
mode_paiement    11
dtype: int64

In [351]:
# Traitez les valeurs manquantes de categorie, region et mode_paiement

#pour ces valeurs on les fill par non renseigné car ça ne sont pas des valeurs qui ont un fort impact sur le set

data[['categorie','region','mode_paiement']] = data[['categorie','region','mode_paiement']].fillna("Non renseigné")

In [352]:
data.isna().sum()


date_vente       0
produit          0
categorie        0
prix_unitaire    0
quantite         0
client           0
region           0
mode_paiement    0
dtype: int64

In [353]:
#prix_unitaire contient parfois du texte (ex : "20,50 €") — transformez toute la colonne en nombres décimaux

data['prix_unitaire'] = data['prix_unitaire'].str.replace(",",".", regex=False)
data['prix_unitaire'] = data['prix_unitaire'].str.replace("€","", regex=False)
data['prix_unitaire'] = data['prix_unitaire'].str.strip()
data['prix_unitaire'] = data['prix_unitaire'].astype(float)





In [354]:
# date_vente mélange deux formats de date — uniformisez la colonne en type date
data['date_vente'] = pd.to_datetime(data['date_vente'], format='mixed', dayfirst=True)

In [355]:
 #quantite contient des valeurs impossibles (0 ou négatives) — traitez-les, puis convertissez la colonne en entier
data = data[data['quantite']>0]


In [356]:
data.head()

,date_vente,produit,categorie,prix_unitaire,quantite,client,region,mode_paiement
id_vente,,,,,,,,
2,2025-11-09,Casque audio,Electronique,185.41,4.0,Client_1099,Île-de-France,Carte bancaire
3,2025-11-28,Stylo pack x10,fournitures,7.55,1.0,Client_1115,Auvergne-Rhône-Alpes,Espèces
5,2025-10-12,Lampe LED,Mobilier,57.04,1.0,Client_1114,AURA,Chèque
6,2025-05-27,Bureau réglable,MOBILIER,311.61,3.0,Client_1129,IDF,Chèque
7,2025-12-05,Stylo pack x10,Fourniture,7.72,3.0,Client_1165,Occitanie,Virement


In [357]:
#Les colonnes categorie et region contiennent plusieurs orthographes différentes pour une même valeur (majuscules, espaces, abréviations). Normalisez-les pour qu'une même réalité soit toujours écrite de la même façon.

dictionnaire_regions = {
    'NOUVELLE AQUITAINE': 'NOUVELLE-AQUITAINE',
    'NOUVELLE-AQUITAINE': 'NOUVELLE-AQUITAINE',

    'ÎLE-DE-FRANCE': 'ÎLE-DE-FRANCE',
    'ILE DE FRANCE': 'ÎLE-DE-FRANCE',
    'IDF': 'ÎLE-DE-FRANCE',

    'AUVERGNE-RHÔNE-ALPES': 'AUVERGNE-RHÔNE-ALPES',
    'AURA': 'AUVERGNE-RHÔNE-ALPES',

    'OCCITANIE': 'OCCITANIE',

    'BRETAGNE': 'BRETAGNE'
}

dictionnaire_categories = {
    'ÉLECTRONIQUE': 'ÉLECTRONIQUE',
    'ELECTRONIQUE': 'ÉLECTRONIQUE',
    'ELECTRONIQUE ': 'ÉLECTRONIQUE',

    'FOURNITURES': 'FOURNITURES',
    'FOURNITURES ': 'FOURNITURES',
    'FOURNITURE': 'FOURNITURES',

    'MOBILIER': 'MOBILIER'
}

data['categorie'] = data['categorie'].str.upper()
data['categorie'] = data['categorie'].str.strip().replace(dictionnaire_categories)
data['region'] = data['region'].str.upper()
data['region'] = data['region'].str.strip().replace(dictionnaire_regions)


In [358]:
data.head()

,date_vente,produit,categorie,prix_unitaire,quantite,client,region,mode_paiement
id_vente,,,,,,,,
2,2025-11-09,Casque audio,ÉLECTRONIQUE,185.41,4.0,Client_1099,ÎLE-DE-FRANCE,Carte bancaire
3,2025-11-28,Stylo pack x10,FOURNITURES,7.55,1.0,Client_1115,AUVERGNE-RHÔNE-ALPES,Espèces
5,2025-10-12,Lampe LED,MOBILIER,57.04,1.0,Client_1114,AUVERGNE-RHÔNE-ALPES,Chèque
6,2025-05-27,Bureau réglable,MOBILIER,311.61,3.0,Client_1129,ÎLE-DE-FRANCE,Chèque
7,2025-12-05,Stylo pack x10,FOURNITURES,7.72,3.0,Client_1165,OCCITANIE,Virement


In [359]:
data['categorie'].unique()


<StringArray>
['ÉLECTRONIQUE', 'FOURNITURES', 'MOBILIER', 'NON RENSEIGNÉ']
Length: 4, dtype: str

In [360]:
data['region'].unique()


<StringArray>
[       'ÎLE-DE-FRANCE', 'AUVERGNE-RHÔNE-ALPES',            'OCCITANIE',
             'BRETAGNE',   'NOUVELLE-AQUITAINE',        'NON RENSEIGNÉ']
Length: 6, dtype: str

In [361]:
# Créez une colonne montant_total = prix_unitaire × quantite

data['montant_total'] = data['prix_unitaire']*data['quantite']

In [362]:
data.head()

,date_vente,produit,categorie,prix_unitaire,quantite,client,region,mode_paiement,montant_total
id_vente,,,,,,,,,
2,2025-11-09,Casque audio,ÉLECTRONIQUE,185.41,4.0,Client_1099,ÎLE-DE-FRANCE,Carte bancaire,741.64
3,2025-11-28,Stylo pack x10,FOURNITURES,7.55,1.0,Client_1115,AUVERGNE-RHÔNE-ALPES,Espèces,7.55
5,2025-10-12,Lampe LED,MOBILIER,57.04,1.0,Client_1114,AUVERGNE-RHÔNE-ALPES,Chèque,57.04
6,2025-05-27,Bureau réglable,MOBILIER,311.61,3.0,Client_1129,ÎLE-DE-FRANCE,Chèque,934.83
7,2025-12-05,Stylo pack x10,FOURNITURES,7.72,3.0,Client_1165,OCCITANIE,Virement,23.16


In [363]:
data.to_csv("ventes_clear.csv")